# Temporal Structure Analysis

**Objective**: Investigate date quality in the dataset to identify:
1. Records with missing/invalid dates
2. Outlier dates (e.g., 1940s when conflict started in ~1988)
3. Optimal date filtering strategy for model training

## 1. Initial Temporal Distribution

In [1]:
# Quick temporal analysis to determine 80/10/10 split dates
import pandas as pd

# Load the local augmented dataset (has date column)
data = pd.read_csv('../../data/location_info_augmented.csv', dtype=str)

print("Available columns in location_info_augmented.csv:")
print(data.columns.tolist())
print(f"\nFirst few rows:")
print(data.head())

# Convert date to datetime and remove timezone info for easier comparison
data['date'] = pd.to_datetime(data['date'], errors='coerce').dt.tz_localize(None)

# Remove rows with invalid dates
data_with_dates = data[data['date'].notna()].copy()

print("\n" + "="*80)
print("TEMPORAL DISTRIBUTION ANALYSIS")
print("="*80)
print(f"\nTotal incidents: {len(data)}")
print(f"Incidents with valid dates: {len(data_with_dates)} ({len(data_with_dates)/len(data)*100:.1f}%)")
print(f"Incidents with missing dates: {len(data) - len(data_with_dates)}")

# Sort by date
data_sorted = data_with_dates.sort_values('date').reset_index(drop=True)

# Calculate split points
train_end_idx = int(len(data_sorted) * 0.8)
val_end_idx = int(len(data_sorted) * 0.9)

# Get the dates at split points
train_end_date = data_sorted.iloc[train_end_idx - 1]['date']
val_end_date = data_sorted.iloc[val_end_idx - 1]['date']

print(f"\n" + "="*80)
print("TEMPORAL SPLIT DATES (80/10/10)")
print("="*80)
print(f"\nDate range in dataset:")
print(f"  Earliest: {data_sorted['date'].min().strftime('%Y-%m-%d')}")
print(f"  Latest:   {data_sorted['date'].max().strftime('%Y-%m-%d')}")

print(f"\nProposed splits:")
print(f"  Training set:   {data_sorted['date'].min().strftime('%Y-%m-%d')} to {train_end_date.strftime('%Y-%m-%d')} ({train_end_idx} incidents)")
print(f"  Validation set: {data_sorted.iloc[train_end_idx]['date'].strftime('%Y-%m-%d')} to {val_end_date.strftime('%Y-%m-%d')} ({val_end_idx - train_end_idx} incidents)")
print(f"  Test set:       {data_sorted.iloc[val_end_idx]['date'].strftime('%Y-%m-%d')} to {data_sorted['date'].max().strftime('%Y-%m-%d')} ({len(data_sorted) - val_end_idx} incidents)")

# Check Telangana formation date
telangana_formation = pd.to_datetime('2014-06-02')
print(f"\n" + "="*80)
print("TELANGANA FORMATION DATE: June 2, 2014")
print("="*80)

if train_end_date < telangana_formation:
    years_before = (telangana_formation - train_end_date).days / 365.25
    print(f"✅ Training set ends {years_before:.1f} years BEFORE Telangana formation")
    print(f"   → Model will only see 'Andhra Pradesh' labels during training")
    print(f"   → Test set will have 'Telangana' labels (if incidents are post-2014)")
elif data_sorted.iloc[val_end_idx]['date'] > telangana_formation:
    years_after = (data_sorted.iloc[val_end_idx]['date'] - telangana_formation).days / 365.25
    print(f"✅ Validation set starts {years_after:.1f} years AFTER Telangana formation")
    print(f"   → Clean temporal separation for Telangana vs Andhra Pradesh")
else:
    print(f"⚠️  Telangana formation falls within training/validation period")
    print(f"   → Model will see both 'Andhra Pradesh' and 'Telangana' labels")
    
    # Count how many training examples are pre/post Telangana
    train_data = data_sorted.iloc[:train_end_idx]
    train_pre_telangana = (train_data['date'] < telangana_formation).sum()
    train_post_telangana = (train_data['date'] >= telangana_formation).sum()
    print(f"   → Training set: {train_pre_telangana} pre-2014, {train_post_telangana} post-2014")

# Show distribution by year
print(f"\n" + "="*80)
print("DISTRIBUTION BY YEAR")
print("="*80)
data_sorted['year'] = data_sorted['date'].dt.year
year_counts = data_sorted['year'].value_counts().sort_index()
print(f"\n{year_counts.to_string()}")

# Cumulative distribution
cumulative_pct = year_counts.sort_index().cumsum() / len(data_sorted) * 100
print(f"\n" + "="*80)
print("CUMULATIVE PERCENTAGE BY YEAR")
print("="*80)
for year, pct in cumulative_pct.items():
    marker = ""
    if pct >= 80 and pct < 90:
        marker = " ← 80% cutoff (end of training)"
    elif pct >= 90:
        marker = " ← 90% cutoff (end of validation)"
    print(f"  {year}: {pct:5.1f}%{marker}")

Available columns in location_info_augmented.csv:
['date', 'state', 'district', 'village_name', 'other_locations', 'incident_summary']

First few rows:
         date           state        district  village_name other_locations  \
0  2007-01-01  Andhra Pradesh       Hyderabad           NaN       Cyberabad   
1  2009-01-01  Andhra Pradesh       Nizamabad     Kamareddy             NaN   
2  2006-01-03  Andhra Pradesh         Khammam  Bhadrachalam             NaN   
3  2016-01-05  Andhra Pradesh  Vishakhapatnam           NaN  Visakha Agency   
4  2007-01-06  Andhra Pradesh   Visakhapatnam  Teegalabanda      Pedalavasa   

                                    incident_summary  
0  An alleged arms supplier to the Communist Part...  
1  A Kamareddy dalam (squad) member belonging to ...  
2  Senior CPI-Maoist 'Polit Bureau' and 'central ...  
3  A TDP leader and former Sarpanch of Jerrela Gr...  
4  The CPI-Maoist cadres blasted coffee pulping u...  

TEMPORAL DISTRIBUTION ANALYSIS

Total inci

## 2. Investigate Missing and Outlier Dates

In [2]:
print("="*80)
print("MISSING DATES ANALYSIS")
print("="*80)

# Count missing dates
missing_dates = data['date'].isna().sum()
print(f"\nRecords with missing dates: {missing_dates:,} ({missing_dates/len(data)*100:.1f}%)")
print(f"Records with valid dates: {len(data_with_dates):,} ({len(data_with_dates)/len(data)*100:.1f}%)")

# Show examples of records with missing dates
if missing_dates > 0:
    print(f"\n📋 EXAMPLES OF RECORDS WITH MISSING DATES (first 10):")
    print("-"*80)
    missing_date_records = data[data['date'].isna()].head(10)
    for idx, (i, row) in enumerate(missing_date_records.iterrows(), 1):
        print(f"\n{idx}. State: {row.get('state', 'N/A')}, District: {row.get('district', 'N/A')}")
        print(f"   Village: {row.get('village_name', 'N/A')}")
        if pd.notna(row.get('incident_summary')):
            summary = str(row['incident_summary'])[:120]
            print(f"   Incident: {summary}...")
        
print("\n" + "="*80)

MISSING DATES ANALYSIS

Records with missing dates: 1 (0.0%)
Records with valid dates: 9,915 (100.0%)

📋 EXAMPLES OF RECORDS WITH MISSING DATES (first 10):
--------------------------------------------------------------------------------

1. State: Jharkhand, District: Latehar
   Village: nan
   Incident: Three CPI-Maoist cadres, including a self-styled 'area commander', were arrested when they assembled in a house at Dhara...



In [6]:
print("="*80)
print("OUTLIER DATES ANALYSIS")
print("="*80)

# The Naxalite-Maoist conflict in India intensified in the late 1980s/early 1990s
# Dates before 2005 are likely data entry errors
reasonable_start_date = pd.to_datetime('2005-01-01')

# Count outliers
outliers = data_sorted[data_sorted['date'] < reasonable_start_date]
print(f"\nRecords with dates before 2005: {len(outliers):,} ({len(outliers)/len(data_sorted)*100:.1f}%)")

if len(outliers) > 0:
    print(f"\n📋 OUTLIER DATES (all records before 2005):")
    print("-"*80)
    print(f"\nDate range of outliers: {outliers['date'].min().strftime('%Y-%m-%d')} to {outliers['date'].max().strftime('%Y-%m-%d')}")
    
    # Show all outliers grouped by year
    outlier_years = outliers['date'].dt.year.value_counts().sort_index()
    print(f"\nBreakdown by year:")
    for year, count in outlier_years.items():
        print(f"  {year}: {count} records")
    
    # Show individual records
    print(f"\n📋 DETAILED RECORDS (showing all {len(outliers)} outliers):")
    print("-"*80)
    for idx, (i, row) in enumerate(outliers.iterrows(), 1):
        date_str = row['date'].strftime('%Y-%m-%d')
        print(f"\n{idx}. Date: {date_str}")
        print(f"   State: {row.get('state', 'N/A')}, District: {row.get('district', 'N/A')}")
        print(f"   Village: {row.get('village_name', 'N/A')}")
        if pd.notna(row.get('incident_summary')):
            summary = str(row['incident_summary'])[:150]
            print(f"   Incident: {summary}...")

print("\n" + "="*80)

OUTLIER DATES ANALYSIS

Records with dates before 2005: 1 (0.0%)

📋 OUTLIER DATES (all records before 2005):
--------------------------------------------------------------------------------

Date range of outliers: 1940-09-04 to 1940-09-04

Breakdown by year:
  1940: 1 records

📋 DETAILED RECORDS (showing all 1 outliers):
--------------------------------------------------------------------------------

1. Date: 1940-09-04
   State: Andhra Pradesh, District: Visakhapatnam
   Village: nan
   Incident: Seendri Bathro alias Badri (35), from Kannvaram under Mampa Police Station in Koyyru mandal of Visakhapatnam District was found killed with his neck s...

